In [1]:
import os
from strands.models.gemini import GeminiModel
from dotenv import load_dotenv

load_dotenv()

MODEL_ID = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

model = GeminiModel(
    client_args={
        "vertexai": True,
    },
    model_id=MODEL_ID,
)

print(f"Using {MODEL_ID} on Gemini")

Using gemini-3.5-flash on Gemini


In [2]:
import logging

from strands import Agent
from strands.multiagent import Swarm

# Show each handoff as it happens
logging.getLogger("strands.multiagent").setLevel(logging.DEBUG)
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

In [3]:
level_designer = Agent(
    name="level_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Level Designer in a game design swarm.
Your role is to create level layouts, environments, and visual elements.
Focus ONLY on layout and visual design. Do NOT write narrative or balance difficulty.
After completing your layout, hand off to narrative_designer for story elements
and difficulty_balancer for gameplay tuning.
When receiving feedback from other agents, revise your design accordingly.""",
)

game_designer = Agent(
    name="game_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Game Designer in a game design swarm.
Your role is to playtest designs for fun, usability, and player confusion.
Focus ONLY on gameplay feel and player experience. Do NOT design layouts or write narrative.
When you find issues, hand off to the appropriate specialist to fix them.
When the design is ready, give the final summary and do not hand off.""",
)

narrative_designer = Agent(
    name="narrative_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Narrative Designer in a game design swarm.
Your role is to create story elements, character backstories, and atmospheric details.
Focus ONLY on narrative. Do NOT redesign layouts or adjust difficulty.
After adding narrative elements, hand off to difficulty_balancer for balance review
or game_designer for playtesting feedback.""",
)

difficulty_balancer = Agent(
    name="difficulty_balancer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Difficulty Balancer in a game design swarm.
Your role is to evaluate and tune difficulty progression, enemy placement, and pacing.
Focus ONLY on balance. Do NOT redesign layouts or write narrative.
When you find layout or narrative issues, hand off to the appropriate agent.
After balancing, hand off to game_designer for a final playtest review.""",
)

In [4]:
swarm = Swarm(
    [level_designer, game_designer, narrative_designer, difficulty_balancer],
    entry_point=level_designer,
    max_handoffs=20,
    max_iterations=20,
    execution_timeout=1200.0,
    node_timeout=300.0,
    repetitive_handoff_detection_window=8,
    repetitive_handoff_min_unique_agents=3,
)
print("✓ Swarm created")

DEBUG | strands.multiagent.swarm | nodes=<['level_designer', 'game_designer', 'narrative_designer', 'difficulty_balancer']> | initialized swarm with nodes
DEBUG | strands.multiagent.swarm | entry_point=<level_designer> | configured entry point
DEBUG | strands.multiagent.swarm | tool_count=<1>, node_count=<4> | injected coordination tools into agents


✓ Swarm created


In [5]:
result = swarm("""Design a stealth mission level in an abandoned space station.
The level needs to support multiple playstyles (aggressive stealth, patient stealth, and ghost runs).
Include at least 3 distinct zones with different difficulty levels,
narrative elements that explain why the station was abandoned,
and ensure the difficulty curve is appropriate for mid-game players.""")

print("=" * 80)
print("HANDOFF CHAIN:")
for node in result.node_history:
    print(f"  → {node.node_id}")
print(f"\nStatus: {result.status} | iterations: {result.execution_count} | time: {result.execution_time / 1000:.0f}s")

DEBUG | strands.multiagent.swarm | starting swarm execution
DEBUG | strands.multiagent.swarm | current_node=<level_designer> | starting swarm execution with node
DEBUG | strands.multiagent.swarm | max_handoffs=<20>, max_iterations=<20>, timeout=<1200.0>s | swarm execution config
DEBUG | strands.multiagent.swarm | current_node=<level_designer>, iteration=<1> | executing node
DEBUG | strands.multiagent.swarm | from_node=<level_designer>, to_node=<narrative_designer> | handing off from agent to agent
DEBUG | strands.multiagent.swarm | node=<level_designer> | node execution completed
DEBUG | strands.multiagent.swarm | from_node=<level_designer>, to_node=<narrative_designer> | handoff detected
DEBUG | strands.multiagent.swarm | current_node=<narrative_designer>, iteration=<2> | executing node
DEBUG | strands.multiagent.swarm | from_node=<narrative_designer>, to_node=<difficulty_balancer> | handing off from agent to agent
DEBUG | strands.multiagent.swarm | node=<narrative_designer> | node ex

HANDOFF CHAIN:
  → level_designer
  → narrative_designer
  → difficulty_balancer
  → game_designer
  → difficulty_balancer
  → game_designer

Status: Status.COMPLETED | iterations: 6 | time: 155s


In [6]:
from IPython.display import Markdown, display

last_node = result.node_history[-1].node_id
display(Markdown(str(result.results[last_node].result)))

### Playtest Review & Gameplay Feel Assessment
**Subject:** Aegis-IV Research Station Stealth Mission (Final Review)  
**Tester Persona:** Game Designer (Focus: Pacing, Player Feel, Usability, Tension)

The latest balance adjustments have successfully resolved the previous issues of hazard progression inversion and Zone 1 downtime. I have conducted a simulated playtest focusing on **pacing, tension, usability, and playstyle execution** across all three zones. 

Below is the gameplay feel assessment and playtest report.

---

### 1. Zone-by-Zone Playtest Experience

#### **Zone 1: Introductory (Warmup Zone)**
*   **The Feel:** "Active but Highly Forgiving."
*   **Pacing & Tension:** The previous 45–60s patrol loop caused players to sit behind crates waiting, destroying momentum. Tightening this to **20–30s** feels fantastic. It keeps players engaged and lets them plan their moves quickly. 
*   **Usability (Hazards & Alarms):** The **12s hazard warmup** and **10s alarm delay** act as a perfect "tutorial buffer." If a new player makes a mistake, they aren't instantly punished. They have plenty of time to step out of the hazard (which only deals a minor 5 HP/sec) or duck back behind cover to break line-of-sight before the base-wide alarm triggers. 

#### **Zone 2: Intermediate (The Sweet Spot)**
*   **The Feel:** "Calculated Risk and Escalation."
*   **Pacing & Tension:** With **5 Guards and 2 Drones** on a **25–35s loop**, the level layout starts to feel like a real stealth puzzle. Sightlines begin to overlap. 
*   **Usability (Hazards & Alarms):** The **7s hazard warmup** and **15 HP/sec damage** introduce a genuine threat level. Players can no longer ignore hazards, but they still have a comfortable safety margin to correct errors. The **7s alarm delay** requires players to act with intent—if spotted, they must move immediately, but they aren't forced into frantic panic.

#### **Zone 3: Advanced (High Stakes / Intensity)**
*   **The Feel:** "Adrenaline-Fueled Clutches."
*   **Pacing & Tension:** A **12–18s patrol loop** with heavy units (**4 Mechs, 4 Sentries**) creates an incredibly tight, dynamic environment. The frequency of enemy movement keeps the player's heart rate high.
*   **Usability (Hazards & Alarms):** The **4s hazard warmup** combined with **35 HP/sec damage** means entering a hazard is a high-consequence event. If caught inside, players have roughly **2.8 seconds of exposure** to escape before dying. This creates a brilliant "clutch" moment where players must use dash or defensive abilities to survive. 
*   The **5s alarm delay** is the MVP change here; it perfectly supports the "Aggressive Stealth" loop, allowing a player to sprint out of cover, execute a high-risk takedown, or throw an EMP to shut down a Mech before the lockdown triggers.

---

### 2. Playstyle Compatibility Assessment

*   **The Patient Stealth Player (The "Planner"):**
    *   *Feedback:* Highly satisfying. The tightened patrol loops in Zone 1 (20–30s) and Zone 2 (25–35s) mean these players spend less time waiting and more time observing predictable pathing. They can cleanly map out routes in their heads and execute them with confidence.
*   **The Ghost Player (The "Shadow"):**
    *   *Feedback:* Zone 3 is a Ghost runner's dream. Bypassing 4 Mechs and 4 Sentries on a rapid 12–18s rotation requires millimeter-precise movement and perfect timing. The clear warning indicators of the 4s hazard warmup allow them to dance right on the edge of danger without breaking stealth.
*   **The Aggressive Stealth Player (The "Panther"):**
    *   *Feedback:* Outstanding recovery mechanics. If spotted in Zone 3, the 5-second alarm delay window turns a potential "instant-fail" scenario into an action-movie recovery. They have just enough time to slide-tackle a guard, fire a silenced weapon, or throw a smoke grenade to reset the detection meter.

---

### Final Playtest Verdict: **APPROVED FOR PRODUCTION**

The difficulty curve is beautifully tuned for a mid-game level. It successfully balances tension, accessibility, and high-skill expression without causing player confusion or frustration. 

*   **Zone 1** successfully teaches the rules.
*   **Zone 2** tests the player's execution of those rules.
*   **Zone 3** demands mastery but gives the player the mechanical tools to recover when they fail.

No further balance passes, layout adjustments, or narrative tweaks are required. The Aegis-IV Research Station level is ready for implementation!
